In [5]:
import os
from getpass import getpass

GEMINI_API_KEY = getpass("Enter your Gemini API key: ")

llm_config = {
    "config_list": [
        {
            "model": "gemini-3.5-flash-lite",
            "api_type": "google",
            "api_key": GEMINI_API_KEY
        }
    ]
}
print("GEMINI configuration created successfully!")

Enter your Gemini API key:  ········


GEMINI configuration created successfully!


In [6]:
from autogen import ConversableAgent

onboarding_personal_information_agent = ConversableAgent(
    name="onboarding_personal_information_agent",  # Removed spaces
    system_message='''You are a helpful customer onboarding agent.
    You are here to help new customers get started with our product.
    Your job is to gather customer's name and location.
    Do not ask for other information. Return 'TERMINATE' 
    when you have gathered all the information.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
)

onboarding_topic_preference_agent = ConversableAgent(
    name="onboarding_topic_preference_agent",  # Removed spaces
    system_message='''You are a helpful customer onboarding agent.
    You are here to help new customers get started with our product.
    Your job is to gather customer's preferences on news topics.
    Do not ask for other information.
    Return 'TERMINATE' when you have gathered all the information.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
)

customer_engagement_agent = ConversableAgent(
    name="customer_engagement_agent",  # Removed spaces
    system_message='''You are a helpful customer service agent.
    You are here to provide fun for the customer based on the user's
    personal information and topic preferences.
    This could include fun facts, jokes, or interesting stories.
    Make sure to make it engaging and fun!
    Return 'TERMINATE' when you are done.''',
    llm_config=llm_config,
    code_execution_config=False,
    human_input_mode="NEVER",
    is_termination_msg=lambda msg: "terminate" in msg.get("content").lower(),
)

customer_proxy_agent = ConversableAgent(
    name="customer_proxy_agent",
    llm_config=False,
    code_execution_config=False,
    human_input_mode="ALWAYS",
    is_termination_msg=lambda msg: "terminate" in msg.get("content").lower(),
)


Creating tasks
Now, you can craft a series of tasks to facilitate the onboarding process.

In [7]:
chats = [
    {
        "sender": onboarding_personal_information_agent,
        "recipient": customer_proxy_agent,
        "message": "Hello, I'm here to help you get started with our product. Could you tell me your name and location?",
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Return the customer information as JSON object only: {'name': '', 'location': ''}",
        },
        "max_turns": 1,
        "clear_history": True
    },
    {
        "sender": onboarding_topic_preference_agent,
        "recipient": customer_proxy_agent,
        "message": "Great! Could you tell me what topics you are interested in reading about?",
        "summary_method": "reflection_with_llm",
        "max_turns": 1,
        "clear_history": False
    },
    {
        "sender": customer_proxy_agent,
        "recipient": customer_engagement_agent,
        "message": "Let's find something fun to read.",
        "max_turns": 1,
        "summary_method": "reflection_with_llm",
    },
]

Start the onboarding process
Note: You might get a slightly different response than what's shown in the video. Feel free to try different inputs, such as name, location, and preferences.

In [8]:
from autogen import initiate_chats

chat_results = initiate_chats(chats)


********************************************************************************
Starting a new chat....

********************************************************************************
onboarding_personal_information_agent (to customer_proxy_agent):

Hello, I'm here to help you get started with our product. Could you tell me your name and location?

--------------------------------------------------------------------------------


Replying as customer_proxy_agent. Provide feedback to onboarding_personal_information_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  Sathish from Ulundurpet


customer_proxy_agent (to onboarding_personal_information_agent):

Sathish from Ulundurpet

--------------------------------------------------------------------------------

********************************************************************************
Starting a new chat....

********************************************************************************
onboarding_topic_preference_agent (to customer_proxy_agent):

Great! Could you tell me what topics you are interested in reading about?
Context: 
```json
{
  "name": "Sathish",
  "location": "Ulundurpet"
}
```

--------------------------------------------------------------------------------


Replying as customer_proxy_agent. Provide feedback to onboarding_topic_preference_agent. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  mobile service


customer_proxy_agent (to onboarding_topic_preference_agent):

mobile service

--------------------------------------------------------------------------------


C:\Anaconda3\envs\agenticAI\Lib\site-packages\autogen\oai\gemini.py:809: UserWarning: Cost calculation is not implemented for model gemini-3.5-flash-lite. Cost will be calculated zero.
  warnings.warn(



********************************************************************************
Starting a new chat....

********************************************************************************
customer_proxy_agent (to customer_engagement_agent):

Let's find something fun to read.
Context: 
```json
{
  "name": "Sathish",
  "location": "Ulundurpet"
}
```
Sathish from Ulundurpet is interested in reading about mobile service topics.

--------------------------------------------------------------------------------
customer_engagement_agent (to customer_proxy_agent):

Hello Sathish! It is fantastic to connect with someone from Ulundurpet today! 📱✨

Since I know you're super interested in mobile services, I dug up a fun little fact and a joke just for you to brighten up your day:

**Did you know?** 
The very first mobile phone call in history was made on April 3, 1973, by a man named Martin Cooper, a senior engineer at Motorola. He stood on a street in Manhattan and called his main rival at Bell L

In [13]:
summary=[]
for chat_result in chat_results:
    print(chat_result.summary)
    summary.append(chat_result.summary)
    print(" ")

{'content': '```json\n{\n  "name": "Sathish",\n  "location": "Ulundurpet"\n}\n```', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}
 
{'content': 'Sathish from Ulundurpet is interested in reading about mobile service topics.', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}
 
{'content': 'Sathish from Ulundurpet, who is interested in mobile services, was entertained with a historical fact about the first mobile phone call made in 1973 and a mobile-themed joke.', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}
 


In [17]:
summary[0]

{'content': '```json\n{\n  "name": "Sathish",\n  "location": "Ulundurpet"\n}\n```',
 'refusal': None,
 'role': 'assistant',
 'annotations': None,
 'audio': None,
 'function_call': None,
 'tool_calls': None}

In [18]:
import pandas as pd
import json
import re

# Extract content from the dictionary
data_str = summary[0]['content']

# Remove Markdown code fences
data_str = re.sub(r'```json\s*|\s*```', '', data_str).strip()

# Convert JSON string into Python dictionary
data_dict = json.loads(data_str)

# Convert dictionary into DataFrame
df = pd.DataFrame([data_dict])

df

,name,location
0,Sathish,Ulundurpet


In [26]:
df["TopicPreference"]=summary[1]['content']

In [28]:
df["Enagement"]=summary[2]['content']
df

,name,location,TopicPreference,Enagement
0,Sathish,Ulundurpet,Sathish from Ulundurpet is interested in readi...,"Sathish from Ulundurpet, who is interested in ..."


In [29]:
def add_leads_to_google_sheet(df):
    import gspread
    from oauth2client.service_account import ServiceAccountCredentials

    # Define the scope for Google Sheets API
    scope = ['https://www.googleapis.com/auth/spreadsheets']

    # Add credentials to the account
    creds = ServiceAccountCredentials.from_json_keyfile_name(
        'customer-onboarding-openai-97aa3c132d90.json', scope
    )

    # Authorize the client
    client = gspread.authorize(creds)

    # Specify the Google Sheet ID
    sheet_id = '1715fbc-io2EH8snyJRxbbCqE-6pKwonodUoFPHr-r38'

    # Open the spreadsheet using its ID
    spreadsheet = client.open_by_key(sheet_id)

    # Select the first worksheet (index 0)
    worksheet = spreadsheet.get_worksheet(0)

    # Append each row of the DataFrame to the worksheet
    for _, row in df.iterrows():
        # Convert row to a list and append to the sheet
        worksheet.append_row(row.tolist())

    print("Data added successfully!")
    return "added"


In [30]:
add_leads_to_google_sheet(df)

Data added successfully!


'added'

Print out the cost

In [31]:
for chat_result in chat_results:
    print(chat_result.cost)
    print("\n")

{'usage_including_cached_inference': {'total_cost': 0, 'gemini-3.5-flash-lite': {'cost': 0, 'prompt_tokens': 52, 'completion_tokens': 28, 'total_tokens': 80}}, 'usage_excluding_cached_inference': {'total_cost': 0}}


{'usage_including_cached_inference': {'total_cost': 0, 'gemini-3.5-flash-lite': {'cost': 0, 'prompt_tokens': 68, 'completion_tokens': 16, 'total_tokens': 84}}, 'usage_excluding_cached_inference': {'total_cost': 0, 'gemini-3.5-flash-lite': {'cost': 0, 'prompt_tokens': 68, 'completion_tokens': 16, 'total_tokens': 84}}}


{'usage_including_cached_inference': {'total_cost': 0, 'gemini-3.5-flash-lite': {'cost': 0, 'prompt_tokens': 469, 'completion_tokens': 302, 'total_tokens': 771}}, 'usage_excluding_cached_inference': {'total_cost': 0, 'gemini-3.5-flash-lite': {'cost': 0, 'prompt_tokens': 469, 'completion_tokens': 302, 'total_tokens': 771}}}


